# MedSAM Full Pipeline (Exact Execution)

# Imports

In [ ]:

import os
import json
import cv2
import numpy as np
import torch
from pathlib import Path
from collections import defaultdict
import re


# Configuration

In [ ]:

DATASORT_DIR = Path.cwd()
GT_PATH = DATASORT_DIR / "ground_truth.json"
MEDSAM_OUT_DIR = DATASORT_DIR / "medsam_output"

CLASSES = ['Ulcer', 'Erosion', 'Blood - fresh', 'Polyp']


# Bounding Box Utilities

In [ ]:

def shape_to_bbox(shape):
    xs = [pt["x"] for pt in shape]
    ys = [pt["y"] for pt in shape]
    return [min(xs), min(ys), max(xs), max(ys)]


# Mask Processing

In [ ]:

def mask_to_bbox(mask_path):
    mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
    if mask is None:
        return None
    coords = cv2.findNonZero(mask)
    if coords is None:
        return None
    x, y, w, h = cv2.boundingRect(coords)
    return {'x1': x, 'y1': y, 'x2': x + w, 'y2': y + h}


# IoU and Dice Metrics

In [ ]:

def iou(boxA, boxB):
    xA = max(boxA['x1'], boxB['x1'])
    yA = max(boxA['y1'], boxB['y1'])
    xB = min(boxA['x2'], boxB['x2'])
    yB = min(boxA['y2'], boxB['y2'])
    inter = max(0, xB - xA) * max(0, yB - yA)
    areaA = (boxA['x2'] - boxA['x1']) * (boxA['y2'] - boxA['y1'])
    areaB = (boxB['x2'] - boxB['x1']) * (boxB['y2'] - boxB['y1'])
    return inter / (areaA + areaB - inter) if inter > 0 else 0


# Ground Truth Loading

In [ ]:

def dice(boxA, boxB):
    xA = max(boxA['x1'], boxB['x1'])
    yA = max(boxA['y1'], boxB['y1'])
    xB = min(boxA['x2'], boxB['x2'])
    yB = min(boxA['y2'], boxB['y2'])
    inter = max(0, xB - xA) * max(0, yB - yA)
    areaA = (boxA['x2'] - boxA['x1']) * (boxA['y2'] - boxA['y1'])
    areaB = (boxB['x2'] - boxB['x1']) * (boxB['y2'] - boxB['y1'])
    return (2*inter)/(areaA+areaB) if (areaA+areaB)>0 else 0


# Mask Extraction

In [ ]:

with open(GT_PATH) as f:
    ground_truth = json.load(f)

gt_lookup = {}
for record in ground_truth:
    vid = record['video_id']
    fid = record['finding_id']
    for bb in record['bboxes']:
        key = (vid, fid, bb['frame'])
        gt_lookup[key] = {'bbox': bb, 'class': record['class']}


# Metric Computation

In [ ]:

masks = []

for class_folder in MEDSAM_OUT_DIR.iterdir():
    if class_folder.name not in CLASSES:
        continue

    for finding_folder in class_folder.iterdir():
        if not finding_folder.is_dir():
            continue

        parts = finding_folder.name.split('_')
        if len(parts) < 2:
            continue

        video_id = parts[0]
        finding_id = parts[1]

        for mask_file in finding_folder.glob("*.png"):
            match = re.search(r'frame_(\d+)', mask_file.stem)
            if not match:
                continue

            frame_num = int(match.group(1))
            pred_bbox = mask_to_bbox(mask_file)

            if pred_bbox:
                masks.append({
                    'video_id': video_id,
                    'finding_id': finding_id,
                    'frame': frame_num,
                    'pred_bbox': pred_bbox,
                    'pred_class': class_folder.name
                })

len(masks)


# Final Results

In [ ]:

TP = defaultdict(int)
FP = defaultdict(int)
FN = defaultdict(int)

iou_scores = []
dice_scores = []
gt_matched = set()

for mask in masks:
    key = (mask['video_id'], mask['finding_id'], mask['frame'])

    if key in gt_lookup:
        gt = gt_lookup[key]
        iou_val = iou(mask['pred_bbox'], gt['bbox'])
        dice_val = dice(mask['pred_bbox'], gt['bbox'])

        if iou_val >= 0.5:
            if mask['pred_class'] == gt['class']:
                TP[gt['class']] += 1
            else:
                FP[mask['pred_class']] += 1
                FN[gt['class']] += 1
            gt_matched.add(key)
        else:
            FP[mask['pred_class']] += 1

        iou_scores.append(iou_val)
        dice_scores.append(dice_val)
    else:
        FP[mask['pred_class']] += 1

for key, gt in gt_lookup.items():
    if key not in gt_matched:
        FN[gt['class']] += 1


In [ ]:

results = {}

for cls in CLASSES:
    tp, fp, fn = TP[cls], FP[cls], FN[cls]

    p = tp/(tp+fp) if tp+fp>0 else 0
    r = tp/(tp+fn) if tp+fn>0 else 0
    f1 = 2*p*r/(p+r) if p+r>0 else 0

    results[cls] = {"precision":p,"recall":r,"f1":f1,"tp":tp,"fp":fp,"fn":fn}

macro_f1 = sum([results[c]["f1"] for c in CLASSES])/len(CLASSES)

total_tp = sum(TP.values())
total_fp = sum(FP.values())
total_fn = sum(FN.values())

micro_p = total_tp/(total_tp+total_fp) if total_tp+total_fp>0 else 0
micro_r = total_tp/(total_tp+total_fn) if total_tp+total_fn>0 else 0
micro_f1 = 2*micro_p*micro_r/(micro_p+micro_r) if micro_p+micro_r>0 else 0

results["macro_f1"] = macro_f1
results["micro_f1"] = micro_f1
results["mean_iou"] = sum(iou_scores)/len(iou_scores) if iou_scores else 0
results["mean_dice"] = sum(dice_scores)/len(dice_scores) if dice_scores else 0

results
